# Using Pretrained Models for NLP

**Today's idea in one line:** someone else already trained a very smart model on millions of texts. We just **download it and use it**, with **no training at all**.

| Step | Use case | Question it answers |
|---|---|---|
| 1 | Sentiment analysis | Is this review positive or negative? |
| 2 | Zero-shot classification | Which topic is this text about? (we choose the topics) |
| 3 | Named Entity Recognition | Which words are names, places, organisations? |
| 4 | Question answering | Can the model find the answer inside a paragraph? |
| 5 | Summarization | Can the model shorten a long text? |
| 6 | Fill-mask | Can the model guess a missing word? |
| 7 | Sentence embeddings | Can we search by *meaning* instead of exact words? |
| 8 | **Nepali** (Devanagari and romanized) | Do these models understand Nepali? |

> Everything we trained on Days 2 to 4 (Bag of Words, TF-IDF, Word2Vec, RNN) started **from zero** using only our 3,000 reviews. Today we start from a model that is already educated.

---
## Part 0: Setup

### Install (run once in a terminal, not in the notebook)
```
pip install "transformers<5" torch sentence-transformers scikit-learn pandas matplotlib
```
> Use `transformers` **version 4** (`<5`). Version 5 removed some of the pipelines we use today.
>
> **First time? Read `SETUP.md` in this folder** (virtual environment, install, and how to download the models before class).
>
> The first time you use each model, it is **downloaded once** (about 100 MB to 1 GB each, roughly 8 GB in total; Step 8 alone is about 5 GB) and then stored on your computer. After that it loads in seconds, even offline. You need internet for the first run only.

> **Note:** the first line of the next cell tells Hugging Face to use **PyTorch only** and ignore TensorFlow/Keras (which you may have installed for Day 4). Without it, some computers show a `Keras 3 ... tf-keras` error. It must run **before** `import transformers`.

### What do these packages do?

Before we start, here is what we installed. Only the first three are new today.

| Package | What it is | Real-world analogy | Where we use it |
|---|---|---|---|
| **transformers** | Hugging Face's main library. It downloads pretrained models and gives us `pipeline(...)`. | The **app** that opens the app store and runs the app for us | Every step |
| **torch** (PyTorch) | The engine that does the maths inside the models. We never write PyTorch code ourselves. | The **engine** of a bus: you never touch it, but the bus cannot move without it | Behind every pipeline |
| **sentence-transformers** | Turns a whole sentence into a list of numbers (an embedding) with one line. | A **translator** from sentences into numbers | Step 7 and Step 8.3 (search by meaning) |
| **sentencepiece** and **protobuf** | Helpers that read the tokenizer files of the Nepali models. | The **key** that opens a locked file | Behind the scenes in Step 8 |
| **scikit-learn** | The classic ML library from Days 2 to 4. | An old friend | Train/test split, TF-IDF, Logistic Regression, accuracy |
| **pandas** | Tables (DataFrames). | An old friend | Loading the CSV, showing and saving results |
| **numpy** | Fast arrays and maths. | An old friend | Cosine similarity and search |

> **New today:** transformers, torch, sentence-transformers.
> **Hidden helpers:** sentencepiece, protobuf. We install them but never touch them.
> **Old friends:** scikit-learn, pandas, numpy.

In [5]:
import os
os.environ["USE_TF"] = "0"          # use PyTorch only; ignore TensorFlow/Keras

import time, warnings
import numpy as np
import pandas as pd
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()     # keep the output short

print("transformers", transformers.__version__, "| torch", torch.__version__)
assert int(transformers.__version__.split(".")[0]) < 5, 'Please run: pip install "transformers<5"'

transformers 4.57.6 | torch 2.15.0+cpu


### Load the same 3,000 reviews and the same 600 test reviews as Days 2 to 4
Same `test_size`, `random_state` and `stratify`, so every number today can be compared with what we got earlier.

In [6]:
df = pd.read_csv("Pretrained_Dataset.csv")
df["label"] = (df["sentiment"] == "positive").astype(int)        # positive = 1, negative = 0

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["sentiment"])
print("Train:", len(train_df), "| Test:", len(test_df))
test_df.head(3)

Train: 2400 | Test: 600


,review_id,review,sentiment,label
2676,2677,"What a dissapointing film, I was checking my w...",positive,1
2030,2031,"Best movie ever, the ending was so satisfying....",positive,1
403,404,"the story was dull, the jokes were flat and th...",negative,0


---
## Part 1: What is a pretrained model?

**Real-world analogy.** Two people apply for a job reading customer reviews.
- **Person A** is a fresh school leaver. You must train them for months with your own examples.
- **Person B** has already read thousands of books, newspapers and reviews. You give them one instruction and they start working on day one.

A **pretrained model** is Person B. Someone (Google, Meta, Hugging Face...) trained it on a huge amount of text using powerful computers. We only **download** it.

| | Models we built (Days 2 to 4) | Pretrained model |
|---|---|---|
| Starts from | nothing | already trained on huge data |
| Needs our data to learn | yes, 2,400 reviews | no |
| Code | `Sequential`, `compile`, `fit` ... | 2 to 3 lines |
| Training time | seconds to minutes | **none** |

> Remember GloVe and Word2Vec? Those were also *pretrained* word meanings. Today we use bigger, smarter ones that understand whole sentences.

### Two new tools: Hugging Face and `pipeline`

**Hugging Face** is like an **app store for AI models**. Instead of apps, it hosts thousands of ready-made models that anybody can download for free.

A **pipeline** is a ready-made machine with three parts inside. You only touch the two ends:

```
your text  ->  [ 1. tokenizer: text to numbers  ->  2. model: thinks  ->  3. answer: numbers back to words ]  ->  result
```

**Example of the idea (do not run):** `pipeline("sentiment-analysis")` means *"give me the sentiment-analysis machine"*. Then `machine("I loved it")` gives the answer.

---
## Step 1: Sentiment analysis (the task we already know)

**Real-world analogy.** Reading a Facebook comment under a movie trailer and deciding "this person liked it" or "this person did not".

We ask Hugging Face for a *sentiment-analysis* machine. We also write the model name, so everybody in class gets exactly the same one.

In [7]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
print("Model loaded!")

Model loaded!


Now use it on **one** sentence. First look at the raw output, then pick the parts we want.

In [8]:
result = sentiment("The acting was wonderful and the story touched my heart.")
print(result)                 # a list with one dictionary

[{'label': 'POSITIVE', 'score': 0.9998884201049805}]


In [9]:
print("Label:", result[0]["label"])
print("Score:", round(result[0]["score"], 3))      # how sure the model is (1.0 = fully sure)

Label: POSITIVE
Score: 1.0


It also accepts a **list** of sentences. Try a few of our own, including a negation (the thing that confused Bag of Words).

In [10]:
sentences = ["I loved this movie!",
             "The movie was not good.",
             "The editing was messy and the pacing was awful."]

for s, r in zip(sentences, sentiment(sentences)):
    print(f"{r['label']:9} {r['score']:.3f}   {s}")

POSITIVE  1.000   I loved this movie!
NEGATIVE  1.000   The movie was not good.
NEGATIVE  1.000   The editing was messy and the pacing was awful.


### Now the real test: our 600 test reviews
We did **not** train on our data at all. The model has never seen these reviews. We simply ask it for a prediction for each one.

In [11]:
start = time.time()
raw = sentiment(test_df["review"].tolist(), batch_size=32, truncation=True)
pretrained_seconds = time.time() - start

pred_pretrained = np.array([1 if r["label"] == "POSITIVE" else 0 for r in raw])
acc_pretrained = accuracy_score(test_df["label"], pred_pretrained)
print("Pretrained model accuracy:", round(acc_pretrained, 3), "| time:", round(pretrained_seconds, 1), "s")

Pretrained model accuracy: 0.872 | time: 29.2 s


For fairness, here is a simple **Day 2 style** model (TF-IDF + Logistic Regression) on the same split. This one *does* need training.

In [9]:
start = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
lr = LogisticRegression(max_iter=1000).fit(tfidf.fit_transform(train_df["review"]), train_df["label"])
lr_seconds = time.time() - start

pred_lr = lr.predict(tfidf.transform(test_df["review"]))
acc_lr = accuracy_score(test_df["label"], pred_lr)

pd.DataFrame({"Method": ["TF-IDF + LogReg (trained by us)", "Pretrained DistilBERT (not trained by us)"],
              "Accuracy": [round(acc_lr, 3), round(acc_pretrained, 3)],
              "Training needed?": ["yes, " + str(round(lr_seconds, 1)) + " s", "no"]})

,Method,Accuracy,Training needed?
0,TF-IDF + LogReg (trained by us),0.930,"yes, 0.2 s"
1,Pretrained DistilBERT (not trained by us),0.872,no


**Discuss (honestly!):** The pretrained model never saw our reviews, yet it gets most of them right. But the model we trained **ourselves** scored a little higher. Why?
- Our reviews are short, simple and all from the same style. A model trained on exactly this style learns it very well.
- Our teaching dataset also contains a few **deliberately wrong labels** (like real data), so nobody can reach 100%.
- The pretrained model needed **no training data and no training time**, and it can also work on text from *any* topic. The model we trained only knows our reviews.

**Rule of thumb:** with plenty of clean data of your own, a trained model can win. With little or no data, a pretrained model wins easily.

### Where does the pretrained model make mistakes?
Always read the mistakes. Let us look at five.

In [12]:
errors = test_df[pred_pretrained != test_df["label"].values].head(5)
for _, row in errors.iterrows():
    print("True label:", row["sentiment"], "|", row["review"][:150])
    print()

True label: positive | What a dissapointing film, I was checking my watch the whole time. ugh, boring af

True label: positive | The hero was good. The camera work was brilliant in places. pls avoid this one. The jokes were not good. The leading actors were not convincing at all

True label: positive | @filmfan21 The villain was weak. Not bad at all, I did not expect to like it this much. gr8 movie lol

True label: negative | A heartwarming story that stays with you long after the credits. The villain was weak. No complaints at all, it was great

True label: negative | <p>The villain was weak. I never felt bored, the story was gripping!!</p>



Read them one by one. In several of them, is the **model** really wrong, or does the **label** look wrong? Real data is never perfectly clean, and no model can be 100%.

---
## Step 2: Zero-shot classification (you choose the labels today)

**Real-world analogy.** On your first day at a post office the manager hands you three boxes labelled *Kathmandu*, *Pokhara*, *Biratnagar*. Nobody trained you on these exact labels. You read each letter's address and put it in the right box.

**Zero-shot** means the model sorts texts into categories it was **never trained on**. We give the category names at run time.

In [13]:
zero_shot = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-3")

headline = "Nepal beat India by 5 wickets in a thrilling final at Tribhuvan University ground."
labels = ["sports", "politics", "health", "technology"]

out = zero_shot(headline, candidate_labels=labels)
print(out)

{'sequence': 'Nepal beat India by 5 wickets in a thrilling final at Tribhuvan University ground.', 'labels': ['sports', 'technology', 'health', 'politics'], 'scores': [0.8837694525718689, 0.0656081885099411, 0.02701469697058201, 0.023607710376381874]}


In [14]:
for label, score in zip(out["labels"], out["scores"]):          # already sorted, best first
    print(f"{label:12} {score:.3f}")

sports       0.884
technology   0.066
health       0.027
politics     0.024


Same machine, **new labels**. Imagine you run an online shop and want to sort customer messages into the right department.

In [15]:
departments = ["delivery", "payment", "product quality"]

messages = ["Where is my parcel? It was supposed to arrive last week.",
            "I was charged twice for the same order.",
            "The phone screen broke after two days."]

for m in messages:
    r = zero_shot(m, candidate_labels=departments)
    print(f"{r['labels'][0]:16} ({r['scores'][0]:.2f})   {m}")

delivery         (0.88)   Where is my parcel? It was supposed to arrive last week.
payment          (0.69)   I was charged twice for the same order.
product quality  (0.84)   The phone screen broke after two days.


---
## Step 3: Named Entity Recognition (NER)

**Real-world analogy.** A news editor reads an article with three coloured highlighters: yellow for **people**, green for **places**, blue for **organisations**.

NER does the same highlighting automatically. It finds the names inside a sentence and tells us the type: `PER` (person), `LOC` (location), `ORG` (organisation), `MISC` (other).

### Where do `PER`, `ORG` and `LOC` come from?

These labels were **not chosen by our code**. They come from the model's training.

The people who built this model trained it on a dataset called **CoNLL-2003**: news articles in which humans had underlined every name by hand and written its type next to it:

| Label | Meaning | Example |
|---|---|---|
| `PER` | person | Ram Sharma |
| `ORG` | organisation | Tribhuvan University, Nepal Telecom |
| `LOC` | location | Kathmandu |
| `MISC` | other names | (for example a nationality or an event) |

**Real-world analogy.** Different forms have different columns. A bank form asks for *account number* and *PAN*; a school form asks for *class* and *roll number*. In the same way, **different NER models use different labels**, depending on the data they were trained on.

> A model can only find the types it was taught. This one cannot find **dates**, **money** or **phone numbers**. Some other models can, because they were trained on data with those labels.

### Other NER label sets (just so you know they exist)

Different NER models are trained on different datasets, so they find **different types of names**. Only the first row is the one we ran today.

| Training data | Labels | Plain-language idea | Example |
|---|---|---|---|
| **CoNLL-2003** (our model) | `PER`, `ORG`, `LOC`, `MISC` | The basic set: people, organisations, places, other | Ram Sharma, Nepal Telecom, Kathmandu |
| **WikiANN** (many languages) | `PER`, `ORG`, `LOC` | The same basic idea, used by many multilingual models | राम शर्मा, त्रिभुवन विश्वविद्यालय |
| **OntoNotes** (18 types) | `PERSON`, `ORG`, `GPE` (countries and cities), `LOC` (mountains, rivers), `NORP` (nationalities and groups), `FAC` (buildings, airports), `DATE`, `TIME`, `MONEY`, `PERCENT`, `QUANTITY`, `EVENT`, `PRODUCT`, `LANGUAGE`, `LAW`, `WORK_OF_ART`, `ORDINAL`, `CARDINAL` | A much richer set that also finds dates, money and numbers | Nepal (`GPE`), Mount Everest (`LOC`), 12 October (`DATE`), Rs 5,000 (`MONEY`) |
| **Medical** | `DISEASE`, `CHEMICAL`, `GENE`... | Trained on medical papers | dengue, paracetamol |
| **Built for one job** | whatever the builder chose, for example `SKILL`, `JOB_TITLE`, `PRICE`, `PHONE` | Companies train their own for CVs, shops or legal documents | Python (`SKILL`), Rs 1,200 (`PRICE`) |

In [17]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="first")

text = "Ram Sharma studies at Tribhuvan University in Kathmandu and works for Nepal Telecom."
for e in ner(text):
    print(f"{e['word']:25} {e['entity_group']:5} {e['score']:.2f}")

ValueError: Could not load model dslim/bert-base-NER with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForTokenClassification'>, <class 'transformers.models.bert.modeling_bert.BertForTokenClassification'>). See the original errors:

while loading with AutoModelForTokenClassification, an error is thrown:
Traceback (most recent call last):
  File "E:\python\broadways python\NLP\Pre Trained Models\nlp-venv\Lib\site-packages\transformers\pipelines\base.py", line 293, in infer_framework_load_model
    model = model_class.from_pretrained(model, **kwargs)
  File "E:\python\broadways python\NLP\Pre Trained Models\nlp-venv\Lib\site-packages\transformers\models\auto\auto_factory.py", line 604, in from_pretrained
    return model_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *model_args, config=config, **hub_kwargs, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

`aggregation_strategy="first"` glues word-pieces back together (the model cuts a rare name like `Pokhara` into small pieces, and this joins them again). Without it you would get messy pieces.

---
## Step 4: Question answering

**Real-world analogy.** An **open-book exam**. You are given a paragraph and a question, and you must **underline the answer inside the paragraph**.

The model does not know the answer by memory. It *finds* it in the text we give.

In [18]:
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

context = ("Mount Everest is the highest mountain in the world. It is located in the Himalayas on the border "
           "between Nepal and China. Its height is 8,849 metres. Tenzing Norgay and Edmund Hillary were the "
           "first climbers to reach the summit, in 1953.")

for question in ["How high is Mount Everest?",
                 "Who first climbed Mount Everest?",
                 "When was the summit first reached?"]:
    a = qa(question=question, context=context)
    print(f"Q: {question}\nA: {a['answer']}   (confidence {a['score']:.2f})\n")

ValueError: Could not load model distilbert-base-cased-distilled-squad with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForQuestionAnswering'>, <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForQuestionAnswering'>). See the original errors:

while loading with AutoModelForQuestionAnswering, an error is thrown:
Traceback (most recent call last):
  File "E:\python\broadways python\NLP\Pre Trained Models\nlp-venv\Lib\site-packages\transformers\pipelines\base.py", line 293, in infer_framework_load_model
    model = model_class.from_pretrained(model, **kwargs)
  File "E:\python\broadways python\NLP\Pre Trained Models\nlp-venv\Lib\site-packages\transformers\models\auto\auto_factory.py", line 604, in from_pretrained
    return model_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *model_args, config=config, **hub_kwargs, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

**What if the answer is not in the paragraph?** Watch the confidence score.

In [19]:
a = qa(question="Who is the Prime Minister of Japan?", context=context)
print("A:", a["answer"], "  (confidence", round(a["score"], 3), ")")

NameError: name 'qa' is not defined

It still gives *some* answer, but the confidence is **much lower** than before (about 0.35, against 0.93 to 0.98). A low score is the model telling us: *I am guessing*. Never trust it blindly.

---
## Step 5: Summarization

**Real-world analogy.** Your teacher gives you a 5-page chapter the night before the exam and says *"write the key points in 2 lines"*.

This model **reads** a long text and **writes** a shorter version.

In [16]:
summarizer = pipeline("summarization", model="t5-small")

article = ("Nepal is a country in South Asia, home to eight of the ten highest mountains in the world, including "
           "Mount Everest. Every year thousands of tourists visit Nepal for trekking, culture and adventure. "
           "Tourism is one of the most important sources of income for the country and gives jobs to many "
           "young people in hotels, restaurants and trekking companies. After the 2015 earthquake, many "
           "heritage sites in Kathmandu valley were rebuilt, and tourist numbers have been growing again.")

summary = summarizer(article, max_length=40, min_length=10)
print(summary[0]["summary_text"])

every year thousands of tourists visit Nepal for trekking, culture and adventure . tourism is one of the most important sources of income for the country .


This is a small model, so it mostly **copies the most important sentences** and shortens them. Bigger models can rewrite in their own words, but they are slower and need a bigger download.

---
## Step 6: Fill-mask (how these models learned)

**Real-world analogy.** A **fill-in-the-blank** question in an exam: *"Kathmandu is the capital of ______."*

This is exactly how BERT-style models were **pretrained**: millions of sentences with a hidden word, and the model had to guess the missing word. After doing this billions of times it learned grammar, facts and meaning, without anyone labelling anything.

In [17]:
fill = pipeline("fill-mask", model="distilbert-base-uncased")

for r in fill("Kathmandu is the capital of [MASK].", top_k=3):
    print(f"{r['token_str']:12} {r['score']:.3f}")

nepal        0.965
kathmandu    0.029
bhutan       0.004


In [18]:
for r in fill("The movie was very [MASK].", top_k=5):
    print(f"{r['token_str']:12} {r['score']:.3f}")

successful   0.362
popular      0.299
profitable   0.033
controversial 0.025
negative     0.018


`[MASK]` marks the hidden word. The model returns its best guesses with a probability for each.

---
## Step 7: Sentence embeddings and semantic search

**Remember Word2Vec (Day 3)?** Every *word* became a list of numbers, and words with similar meaning were close together.

A **sentence embedding** does the same for a **whole sentence**. Every sentence becomes one list of numbers (a vector), and sentences with similar meaning are close together.

**Real-world analogy.** Imagine a library where books are placed on shelves *by meaning*, not by title. A book about "cricket" sits next to a book about "football". If you ask for a book about "games", the librarian walks to that shelf.

We use a new library, **`sentence-transformers`**, which gives us ready-made sentence models.

In [19]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = ["I loved this film.",
             "This movie was fantastic.",
             "The weather in Pokhara is rainy today."]

vectors = embedder.encode(sentences)
print("Shape:", vectors.shape, " -> 3 sentences, each a vector of", vectors.shape[1], "numbers")
print("First 5 numbers of sentence 1:", vectors[0][:5].round(3))

Shape: (3, 384)  -> 3 sentences, each a vector of 384 numbers
First 5 numbers of sentence 1: [-0.079  0.051 -0.001  0.024  0.   ]


### How do we measure "closeness"? Cosine similarity
It gives a score between about 0 (unrelated) and 1 (same meaning). Build it from primitive steps:

In [20]:
a, b, c = vectors

dot = np.dot(a, b)                                   # step 1: multiply and add
length = np.linalg.norm(a) * np.linalg.norm(b)       # step 2: lengths of the two vectors
print("cosine(1, 2) =", round(dot / length, 3))

cosine(1, 2) = 0.794


In [21]:
def cosine(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

print("loved film  vs  fantastic movie :", round(cosine(a, b), 3))   # same meaning, different words
print("loved film  vs  Pokhara weather :", round(cosine(a, c), 3))   # unrelated

loved film  vs  fantastic movie : 0.794
loved film  vs  Pokhara weather : 0.009


### Semantic search: find reviews by **meaning**
We turn all our reviews into vectors once, then compare any question with all of them.

In [22]:
corpus = df.drop_duplicates("review").reset_index(drop=True)        # some reviews repeat; keep one copy of each

start = time.time()
review_vectors = embedder.encode(corpus["review"].tolist(), batch_size=64)
print("Encoded", len(review_vectors), "reviews in", round(time.time() - start, 1), "s | shape:", review_vectors.shape)

Encoded 2960 reviews in 1.8 s | shape: (2960, 384)


In [23]:
def semantic_search(query, top_n=5):
    q = embedder.encode(query)
    scores = review_vectors @ q / (np.linalg.norm(review_vectors, axis=1) * np.linalg.norm(q))   # cosine with every review
    best = np.argsort(scores)[::-1][:top_n]                                                       # highest first
    return pd.DataFrame({"score": scores[best].round(3), "review": corpus["review"].iloc[best].str[:100].values})

semantic_search("a dull movie that wasted my time")

,score,review
0,0.717,"A lazy, predictable and boring film. The story..."
1,0.704,"This movie made me sleepy and annoyed. A lazy,..."
2,0.701,"A lazy, predictable and boring film"
3,0.675,"not a single dull scene, i loved it!!!"
4,0.675,"Not a single dull scene, I loved it!!!"


Now compare with an ordinary **keyword search** (a simple text match) using the same words.

In [24]:
query = "a dull movie that wasted my time"
keyword_hits = corpus[corpus["review"].str.contains(query, case=False)]
print("Reviews that contain the exact text:", repr(query))
print("Number found:", len(keyword_hits))

Reviews that contain the exact text: 'a dull movie that wasted my time'
Number found: 0


Keyword search finds **nothing**, because no review uses those exact words. Semantic search still returned reviews that **mean** the same thing (lazy, boring, sleepy...). That is the power of embeddings.

**But read the last results too:** a review like *"Not a single dull scene, I loved it"* can appear because it talks about the same *topic* (dull). Embeddings capture topic well, but a negation like *not* can still fool them.

---
## Step 8: Pretrained models and the Nepali language

So far every example was in English. What about **Nepali**? There are two ways people write it:
- **Devanagari:** `यो फिल्म धेरै राम्रो थियो।`
- **Romanized** (Nepali typed with English letters, like in Messenger): `Yo film ekdam ramro thiyo.`

**Real-world analogy.** A teacher who only speaks English cannot mark an answer written in Nepali. A teacher who knows many languages can. An **English-only** model is the first teacher. A **multilingual** model (trained on 100+ languages at once) is the second.

First, what happens when we give Nepali to the **English** sentiment model from Step 1?

In [25]:
nepali_reviews = ["यो फिल्म धेरै राम्रो थियो।",           # "This film was very good."
                  "यो फिल्म एकदम बेकार थियो।"]           # "This film was totally useless."

for s, r in zip(nepali_reviews, sentiment(nepali_reviews)):
    print(f"{r['label']:9} {r['score']:.3f}   {s}")

NEGATIVE  0.728   यो फिल्म धेरै राम्रो थियो।
NEGATIVE  0.679   यो फिल्म एकदम बेकार थियो।


The two sentences mean opposite things, yet the English-only model calls **both** negative. It **cannot read Devanagari**, so its answers are meaningless. We need a multilingual model.

### 8.1 Multilingual sentiment
`cardiffnlp/twitter-xlm-roberta-base-sentiment` is built on **XLM-RoBERTa**, a model trained on text from about 100 languages. Same `pipeline`, new model name.

In [26]:
sentiment_multi = pipeline("text-classification", model="cardiffnlp/twitter-xlm-roberta-base-sentiment")

nepali_texts = ["यो फिल्म धेरै राम्रो थियो।",                       # Devanagari, positive
                "यो फिल्म एकदम बेकार थियो।",                       # Devanagari, negative
                "Yo film ekdam ramro thiyo.",                       # romanized, positive
                "Yo film ekdam kharab thiyo, paisa barbad bhayo."]  # romanized, negative

for s, r in zip(nepali_texts, sentiment_multi(nepali_texts)):
    print(f"{r['label']:9} {r['score']:.2f}   {s}")

positive  0.81   यो फिल्म धेरै राम्रो थियो।
negative  0.90   यो फिल्म एकदम बेकार थियो।
positive  0.34   Yo film ekdam ramro thiyo.
negative  0.52   Yo film ekdam kharab thiyo, paisa barbad bhayo.


**Read the scores.** Devanagari is handled with high confidence. Romanized Nepali is much weaker (low scores): spelling in English letters is **not standard** (`ramro`, `ramrro`, `raamro`...) and the model saw far less of it while learning.

### 8.2 Zero-shot in Nepali
The English zero-shot model from Step 2 would fail here too. This one, `mDeBERTa`, was trained on 15 languages for the same job. We give the **labels in English** and the text in Nepali.

In [27]:
zero_shot_multi = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

nepali_messages = ["नेपालले भारतलाई ५ विकेटले हरायो।",       # "Nepal beat India by 5 wickets."
                   "मेरो फोन दुई दिनमै बिग्रियो।"]            # "My phone broke within two days."

for m in nepali_messages:
    r = zero_shot_multi(m, candidate_labels=["sports", "politics", "product quality"])
    print(f"{r['labels'][0]:16} ({r['scores'][0]:.2f})   {m}")

sports           (0.66)   नेपालले भारतलाई ५ विकेटले हरायो।
product quality  (0.77)   मेरो फोन दुई दिनमै बिग्रियो।


### 8.3 Search across languages
Remember sentence embeddings (Step 7)? A **multilingual** embedding model puts sentences with the same meaning close together **even in different languages**. We ask in English and find the answer written in Nepali.

**Real-world analogy.** A librarian who is fluent in both languages puts the English book and the Nepali book about the same topic on the same shelf.

In [28]:
embedder_multi = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

nepali_docs = ["मलाई नेपाली खाना मन पर्छ।",                   # I like Nepali food.
               "आज पोखरामा पानी परिरहेको छ।",                  # It is raining in Pokhara today.
               "काठमाडौं नेपालको राजधानी हो।",                 # Kathmandu is the capital of Nepal.
               "उनले धेरै राम्रो फिल्म हेरे।"]                  # He watched a very good film.

doc_vectors = embedder_multi.encode(nepali_docs)

def nepali_search(query):
    q = embedder_multi.encode(query)
    scores = [cosine(q, d) for d in doc_vectors]          # same cosine() function as before
    best = int(np.argmax(scores))
    print(f"Question: {query}")
    print(f"Best match ({scores[best]:.2f}): {nepali_docs[best]}\n")

nepali_search("Which city is the capital of Nepal?")          # English question
nepali_search("Is it raining today?")
nepali_search("मलाई खाना मन पर्छ")                              # Nepali question

Question: Which city is the capital of Nepal?
Best match (0.48): काठमाडौं नेपालको राजधानी हो।

Question: Is it raining today?
Best match (0.38): आज पोखरामा पानी परिरहेको छ।

Question: मलाई खाना मन पर्छ
Best match (0.63): मलाई नेपाली खाना मन पर्छ।



### 8.4 Translation: Nepali to English
Translation is a **summarization-style** job (read one text, write another). `NLLB` is a Meta model that knows **200 languages**, including Nepali.

> **Heavy model (about 2.5 GB).** This is the biggest download of the day. If your connection is slow, skip this step. The rest of the notebook does not depend on it.

In [29]:
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M", src_lang="npi_Deva", tgt_lang="eng_Latn")

for t in ["मलाई नेपाली खाना मन पर्छ।", "काठमाडौं नेपालको राजधानी हो।", "म विद्यार्थी हुँ र म पोखरामा बस्छु।"]:
    print(translator(t, max_length=60)[0]["translation_text"], "   <-", t)

I like Nepali food.    <- मलाई नेपाली खाना मन पर्छ।
Kathmandu is the capital of Nepal.    <- काठमाडौं नेपालको राजधानी हो।
I am a student and I live in a pond.    <- म विद्यार्थी हुँ र म पोखरामा बस्छु।


Look at the third translation: *"I live in a **pond**"*. The model read `पोखरा` (Pokhara, the city) as `पोखरी` (a pond). Translators can be fooled by **names** that are also ordinary words. Always check the output.

Now the same model on **romanized** Nepali:

In [30]:
print(translator("Ma Kathmandu ma baschu.", max_length=60)[0]["translation_text"])

Ma Kathmandu ma baschu.


It simply copies the text. The model learned Nepali in **Devanagari**, not in romanized form.

### What we learned about Nepali
| | Devanagari | Romanized |
|---|---|---|
| Sentiment | works on our examples | weak |
| Zero-shot | works on our examples | try it yourself (we tested only one) |
| Cross-language search | works on our examples | weak (tested in the background, not shown) |
| Translation (NLLB) | excellent | fails |

**Takeaway:** pretrained multilingual models handle **Devanagari** nicely. For **romanized** Nepali, results are weak, because there is no standard spelling and little training text. This is a real research problem in Nepal.

---
## Putting it together

Every pretrained model we used was one `pipeline(...)` or one `SentenceTransformer(...)` line. Here is our table, **each row with a live example**:

In [31]:
summary_table = pd.DataFrame([
    ["Sentiment",       "sentiment-analysis",         "I loved it!",                            sentiment("I loved it!")[0]["label"]],
    ["Zero-shot",       "zero-shot-classification",   "Goal scored in the last minute",         zero_shot("Goal scored in the last minute", candidate_labels=["sports", "health"])["labels"][0]],
    ["NER",             "ner",                        "Ram lives in Pokhara",                   [(e["word"], e["entity_group"]) for e in ner("Ram lives in Pokhara")]],
    ["Question answer", "question-answering",         "Highest mountain? (Everest paragraph)",  qa(question="Which is the highest mountain?", context=context)["answer"]],
    ["Summarization",   "summarization",              "(long Nepal paragraph)",                 summarizer(article, max_length=20, min_length=5)[0]["summary_text"]],
    ["Fill-mask",       "fill-mask",                  "Dogs are [MASK] animals.",               fill("Dogs are [MASK] animals.", top_k=1)[0]["token_str"]],
], columns=["Use case", "Pipeline name", "Input", "Output"])

summary_table

,Use case,Pipeline name,Input,Output
0,Sentiment,sentiment-analysis,I loved it!,POSITIVE
1,Zero-shot,zero-shot-classification,Goal scored in the last minute,sports
2,NER,ner,Ram lives in Pokhara,"[(Ram, PER), (Pokhara, LOC)]"
3,Question answer,question-answering,Highest mountain? (Everest paragraph),Mount Everest
4,Summarization,summarization,(long Nepal paragraph),every year thousands of tourists visit Nepal f...
5,Fill-mask,fill-mask,Dogs are [MASK] animals.,domestic


### How do I choose a model?
Go to **huggingface.co/models**. On the left pick a *task* (for example "Text Classification"), then sort by *Most downloads*. A model name looks like `dslim/bert-base-NER`:
- `dslim` = who uploaded it
- `bert-base-NER` = what it is. `base` or `small` or `distil` usually means **smaller and faster**, while `large` means **slower but often better**.

> **The same idea works beyond text.** `pipeline("image-classification")` can name what is in a photo, and `pipeline("automatic-speech-recognition")` can turn speech into text. Same pattern, different model.

### Be careful: pretrained does not mean perfect

In [32]:
tricky = ["Oh great, another three hours of my life wasted. Truly a masterpiece.",     # sarcasm
          "The acting was brilliant but the story was boring and far too long."]       # mixed feelings

for s, r in zip(tricky, sentiment(tricky)):
    print(f"{r['label']:9} {r['score']:.3f}   {s}")

POSITIVE  0.991   Oh great, another three hours of my life wasted. Truly a masterpiece.
NEGATIVE  1.000   The acting was brilliant but the story was boring and far too long.


Sarcasm and mixed opinions are hard for humans too. Always test a pretrained model on **your own kind of data** before you trust it.

---
## Your turn!

Run each cell, then **change the text** and run again.

### Exercise 1: Your own sentiment
Write 3 sentences about something you know (a restaurant, a game, a teacher). Try one in Nepali written in English letters. What happens?

In [33]:
my_sentences = ["The momo at this place is delicious.",
                "The service was very slow.",
                "Yo film ekdam ramro thiyo."]          # <- change these

for s, r in zip(my_sentences, sentiment(my_sentences)):
    print(f"{r['label']:9} {r['score']:.3f}   {s}")

POSITIVE  1.000   The momo at this place is delicious.
NEGATIVE  1.000   The service was very slow.
POSITIVE  0.534   Yo film ekdam ramro thiyo.


### Exercise 2: Your own labels (zero-shot)
Pick any text and **any 3 or 4 labels** you like.

In [34]:
my_text = "The bus was late again and the road was full of dust."      # <- change this
my_labels = ["transport", "food", "education"]                          # <- change these

r = zero_shot(my_text, candidate_labels=my_labels)
print(list(zip(r["labels"], [round(x, 2) for x in r["scores"]])))

[('transport', 1.0), ('education', 0.0), ('food', 0.0)]


### Exercise 3: Fill the blank
Write your own sentence with `[MASK]`. Can you make the model say something surprising?

In [35]:
for r in fill("In Nepal, people celebrate [MASK] in autumn.", top_k=5):      # <- change this
    print(f"{r['token_str']:12} {r['score']:.3f}")

christmas    0.091
festivals    0.076
easter       0.043
harvest      0.037
holidays     0.024


### Exercise 4: Search by meaning
Type any description of a review you want to find.

In [36]:
semantic_search("the ending made me cry")          # <- change this

,score,review
0,0.630,THE ENDING WAS BAD BUT THE REST WAS FINE. THE ...
1,0.629,The ending was not satisfying. One song was be...
2,0.625,A heartwarming story that stays with you long ...
3,0.624,The ending was bad but the rest was fine. A he...
4,0.623,The ending was not satisfying. It is not a goo...


### Exercise 5: Your own Nepali sentence
Write one sentence in **Devanagari** and one in **romanized** Nepali. Compare the confidence scores. The romanized one may even give the **wrong** answer.

In [37]:
my_nepali = ["मलाई यो ठाउँ धेरै मन पर्यो।",          # <- change these
             "Malai yo thau dherai man paryo."]

for s, r in zip(my_nepali, sentiment_multi(my_nepali)):
    print(f"{r['label']:9} {r['score']:.2f}   {s}")

positive  0.54   मलाई यो ठाउँ धेरै मन पर्यो।
negative  0.34   Malai yo thau dherai man paryo.


---
## Save our results
Like every project, we keep a reusable output file: our test reviews with the pretrained model's prediction.

In [38]:
# out = test_df[["review_id", "review", "sentiment"]].copy()
# out["pretrained_prediction"] = np.where(pred_pretrained == 1, "positive", "negative")
# out["correct"] = out["sentiment"] == out["pretrained_prediction"]
# out.to_csv("Pretrained_Sentiment_Results.csv", index=False)
# print("Saved", len(out), "rows. Correct:", int(out["correct"].sum()))

---
## Summary: What we learned today

1. A **pretrained model** is already trained by someone else. We **download and use** it, with no `fit()`.
2. **Hugging Face** is the app store for models, and **`pipeline`** is the one-line way to use them.
3. The same idea covers many jobs: **sentiment, zero-shot sorting, NER, question answering, summarization, fill-mask**.
4. **Sentence embeddings** turn sentences into vectors, so we can **search by meaning**.
5. **Multilingual** models can read Nepali in **Devanagari**, but **romanized** Nepali is still weak.
6. Pretrained models are strong but **not perfect**. Always read their mistakes and test on your own data.

> **Next step (not today):** *fine-tuning* means taking a pretrained model and training it a little more on **our own** data to make it even better.